# E-commerce Order Analysis - Exploratory Data Analysis
# 电商订单分析 - 探索性数据分析

This notebook provides an interactive environment for exploring the e-commerce order data.
本笔记本提供了一个交互式环境来探索电商订单数据。

## Table of Contents / 目录

1. [Data Loading / 数据加载](#data-loading)
2. [Data Overview / 数据概览](#data-overview)
3. [Data Cleaning / 数据清洗](#data-cleaning)
4. [Exploratory Analysis / 探索性分析](#exploratory-analysis)
5. [Visualization / 可视化](#visualization)
6. [Insights and Conclusions / 洞察和结论](#insights)

In [ ]:
# Import required libraries
# 导入所需库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-whitegrid')
sns.set_palette('colorblind')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
px.defaults.template = 'plotly_white'
px.defaults.width = 800
px.defaults.height = 600

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')
print('库导入成功!')

## 1. Data Loading / 数据加载 {#data-loading}

Load the e-commerce order data and examine its structure.
加载电商订单数据并检查其结构。

In [ ]:
# Add project root to path
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import project modules
from src.data_utils import DataProcessor
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

# Initialize data processor
processor = DataProcessor()

# Generate sample data if it doesn't exist
raw_data_path = RAW_DATA_DIR / 'orders.csv'
if not raw_data_path.exists():
    print('Generating sample data...')
    print('生成示例数据...')
    processor.generate_sample_data()
    print('Sample data generated successfully!')
    print('示例数据生成成功!')

# Load the data
df = processor.load_data(raw_data_path)
print(f'Data loaded: {len(df)} rows, {len(df.columns)} columns')
print(f'数据已加载: {len(df)} 行, {len(df.columns)} 列')

## 2. Data Overview / 数据概览 {#data-overview}

Examine the basic structure and characteristics of the dataset.
检查数据集的基本结构和特征。

In [ ]:
# Display basic information
print('=== Dataset Info / 数据集信息 ===')
print(f'Shape / 形状: {df.shape}')
print(f'Memory usage / 内存使用: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()

# Display first few rows
print('=== First 5 rows / 前5行 ===')
display(df.head())

# Display data types
print('=== Data Types / 数据类型 ===')
display(df.dtypes.to_frame('Data Type'))

# Display basic statistics
print('=== Basic Statistics / 基本统计 ===')
display(df.describe())

In [ ]:
# Check for missing values
print('=== Missing Values / 缺失值 ===')
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count / 缺失数量': missing_data,
    'Percentage / 百分比': missing_percent
})
display(missing_df[missing_df['Missing Count / 缺失数量'] > 0])

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows / 重复行: {duplicates} ({duplicates/len(df)*100:.2f}%)')

## 3. Data Cleaning / 数据清洗 {#data-cleaning}

Clean and preprocess the data for analysis.
清洗和预处理数据以进行分析。

In [ ]:
# Clean the data
print('Cleaning data...')
print('清洗数据...')

df_clean = processor.clean_data(df)

print(f'Original data: {len(df)} rows')
print(f'Cleaned data: {len(df_clean)} rows')
print(f'Rows removed: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.2f}%)')
print()
print(f'原始数据: {len(df)} 行')
print(f'清洗后数据: {len(df_clean)} 行')
print(f'删除行数: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.2f}%)')

# Display cleaned data info
print('\n=== Cleaned Data Info / 清洗后数据信息 ===')
display(df_clean.head())
display(df_clean.dtypes.to_frame('Data Type'))

## 4. Exploratory Analysis / 探索性分析 {#exploratory-analysis}

Perform exploratory data analysis to understand patterns and trends.
进行探索性数据分析以了解模式和趋势。

In [ ]:
# Import analysis module
from src.analysis import OrderAnalyzer

# Initialize analyzer
analyzer = OrderAnalyzer(df_clean)

# Get basic statistics
basic_stats = analyzer.get_basic_statistics()

print('=== Basic Business Metrics / 基本业务指标 ===')
for key, value in basic_stats.items():
    if isinstance(value, (int, float)):
        if 'amount' in key.lower() or 'value' in key.lower():
            print(f'{key}: ${value:,.2f}')
        else:
            print(f'{key}: {value:,}')
    else:
        print(f'{key}: {value}')

In [ ]:
# Time series analysis
time_analysis = analyzer.analyze_time_series()

print('=== Time Series Insights / 时间序列洞察 ===')

# Daily trends
daily_stats = time_analysis['daily_sales']
print(f'Average daily revenue / 平均日收入: ${daily_stats["total_amount"].mean():,.2f}')
print(f'Peak daily revenue / 最高日收入: ${daily_stats["total_amount"].max():,.2f}')
print(f'Average daily orders / 平均日订单数: {daily_stats["order_count"].mean():.0f}')

# Weekly patterns
weekly_stats = time_analysis['weekly_pattern']
best_day = weekly_stats.loc[weekly_stats['total_amount'].idxmax(), 'order_weekday']
worst_day = weekly_stats.loc[weekly_stats['total_amount'].idxmin(), 'order_weekday']
print(f'Best performing day / 表现最佳的日子: {best_day}')
print(f'Worst performing day / 表现最差的日子: {worst_day}')

In [ ]:
# Customer analysis
customer_analysis = analyzer.analyze_customers()

print('=== Customer Insights / 客户洞察 ===')

# Customer value metrics
customer_value = customer_analysis['customer_value']
print(f'Average customer lifetime value / 平均客户生命周期价值: ${customer_value["total_spent"].mean():,.2f}')
print(f'Top 10% customers contribute / 前10%客户贡献: ${customer_value.nlargest(int(len(customer_value)*0.1), "total_spent")["total_spent"].sum():,.2f}')

# RFM analysis
rfm_analysis = customer_analysis['rfm_analysis']
print('\nRFM Customer Segments / RFM客户细分:')
segment_counts = rfm_analysis['rfm_segment'].value_counts()
for segment, count in segment_counts.items():
    percentage = count / len(rfm_analysis) * 100
    print(f'  {segment}: {count} customers ({percentage:.1f}%)')
    print(f'  {segment}: {count} 客户 ({percentage:.1f}%)')

## 5. Visualization / 可视化 {#visualization}

Create interactive visualizations to explore the data.
创建交互式可视化来探索数据。

In [ ]:
# Sales trend visualization
daily_sales = time_analysis['daily_sales'].reset_index()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Daily Revenue Trend / 日收入趋势', 'Daily Order Count Trend / 日订单数趋势'),
    vertical_spacing=0.1
)

# Revenue trend
fig.add_trace(
    go.Scatter(
        x=daily_sales['order_date'],
        y=daily_sales['total_amount'],
        mode='lines+markers',
        name='Revenue / 收入',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# Order count trend
fig.add_trace(
    go.Scatter(
        x=daily_sales['order_date'],
        y=daily_sales['order_count'],
        mode='lines+markers',
        name='Orders / 订单数',
        line=dict(color='green', width=2)
    ),
    row=2, col=1
)

fig.update_layout(
    title='Sales Performance Over Time / 销售表现时间趋势',
    height=600,
    showlegend=True
)

fig.show()

In [ ]:
# Product category analysis
product_analysis = analyzer.analyze_products()
category_stats = product_analysis['category_analysis']

# Create category performance chart
fig = px.bar(
    category_stats.reset_index(),
    x='product_category',
    y='total_revenue',
    title='Revenue by Product Category / 按产品类别的收入',
    labels={
        'product_category': 'Product Category / 产品类别',
        'total_revenue': 'Total Revenue / 总收入'
    },
    color='total_revenue',
    color_continuous_scale='viridis'
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=500
)

fig.show()

In [ ]:
# Customer demographics
demographics = customer_analysis['demographics']

# Age group distribution
fig = px.pie(
    demographics['age_distribution'].reset_index(),
    values='customer_count',
    names='age_group',
    title='Customer Age Distribution / 客户年龄分布'
)

fig.show()

# Gender distribution
fig = px.pie(
    demographics['gender_distribution'].reset_index(),
    values='customer_count',
    names='customer_gender',
    title='Customer Gender Distribution / 客户性别分布'
)

fig.show()

## 6. Insights and Conclusions / 洞察和结论 {#insights}

Summarize key findings and business insights.
总结关键发现和业务洞察。

### Key Findings / 关键发现

Based on the analysis above, here are the main insights:
基于以上分析，以下是主要洞察：

1. **Sales Performance / 销售表现**
   - [Add your observations about sales trends]
   - [添加您对销售趋势的观察]

2. **Customer Behavior / 客户行为**
   - [Add your observations about customer patterns]
   - [添加您对客户模式的观察]

3. **Product Performance / 产品表现**
   - [Add your observations about product categories]
   - [添加您对产品类别的观察]

### Business Recommendations / 业务建议

1. **Marketing Strategy / 营销策略**
   - [Add marketing recommendations]
   - [添加营销建议]

2. **Inventory Management / 库存管理**
   - [Add inventory recommendations]
   - [添加库存建议]

3. **Customer Retention / 客户保留**
   - [Add customer retention strategies]
   - [添加客户保留策略]

In [ ]:
# Generate comprehensive report
print('Generating comprehensive analysis report...')
print('生成综合分析报告...')

report = analyzer.generate_report()

print('Analysis complete! Check the reports directory for detailed findings.')
print('分析完成！请查看reports目录获取详细结果。')

# Display executive summary
if 'executive_summary' in report:
    print('=== Executive Summary / 执行摘要 ===')
    for key, value in report['executive_summary'].items():
        print(f'{key}: {value}')